#  Toxic Comment Data Loader

prepares and cleans the Jigsaw toxic comment dataset for model training.
- cleans HTML, emojis, whitespace, and URLs
- balances toxic vs non-toxic samples
- logs key values
- returns a balanced and cleaned DataFrame

In [1]:
import pandas as pd
import re
import html
import emoji

TOXIC_LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


##  Step 1: Text Cleaning Function

In [ ]:
def clean_comment(text):
    text = str(text)
    text = html.unescape(text) 
    text = emoji.replace_emoji(text, replace=' [EMOJI] ')  # normalize emoji
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = text.replace("", " ").replace("", " ").replace("	", " ")
    text = re.sub(r"http\S+", "[URL]", text)  
    text = re.sub(r"\s+", " ", text) 
    return text.strip().strip('"').strip("'")


## Step 2: Prepare Balanced Dataset

In [ ]:
def prepare_balanced_dataset(filepath, toxic_labels=TOXIC_LABELS, random_seed=42):
    df = pd.read_csv(filepath)

    print(f"📦 Original dataset shape: {df.shape}")

    # drop missing labels
    df = df.dropna(subset=toxic_labels).copy()

    # clean text
    df['comment_text'] = df['comment_text'].apply(clean_comment)

    # log content analysis
    emoji_count = df['comment_text'].str.contains(r'\[EMOJI\]', regex=True).sum()
    html_count = df['comment_text'].str.contains(r'&[a-z]+;', regex=True).sum()
    url_count = df['comment_text'].str.contains(r'http\S+', regex=True).sum()

    print(f" Comments with [EMOJI]: {emoji_count}")
    print(f" Comments with HTML entities: {html_count}")
    
    print(f" Comments with URLs: {url_count}")

    # balance data
    df["num_labels"] = df[toxic_labels].sum(axis=1)
    toxic_df = df[df["num_labels"] > 0]
    non_toxic_df = df[df["num_labels"] == 0].sample(n=len(toxic_df), random_state=random_seed)

    print(f"✅ Toxic comments: {len(toxic_df)}")
    print(f"✅ Non-toxic sampled: {len(non_toxic_df)}")

    balanced_df = pd.concat([toxic_df, non_toxic_df]).sample(frac=1, random_state=random_seed).reset_index(drop=True)
    print(f"🎯 Final balanced dataset: {len(balanced_df)} rows")

    return balanced_df


## Step 3: Load and Run

In [ ]:
# Load and balance dataset
balanced_df = prepare_balanced_dataset("train.csv")
balanced_df.head()
